# Week 9 Lab: Building and Applying the Exponential Function

Math 140H, Honors Calculus I

This week we do two things Chapter 2 normally would have done for us: we build exp and ln from scratch instead of getting them from an integral, and we put them straight to work modeling physical systems.

How to use this notebook
- Cells marked IN CLASS are the roughly 35 minute core we will do together on Day 3.
- Cells marked HOMEWORK extend the same ideas further and are due before the Week 10 quiz.
- Cells marked PREDICT ask you to write down a guess before running the next code cell. Do not skip these; the point is to notice when your intuition is wrong.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq, curve_fit
from scipy.integrate import solve_ivp

plt.rcParams['figure.figsize'] = (7, 4.5)
plt.rcParams['axes.grid'] = True
rng = np.random.default_rng(140)


## Part 1: Constructing exp from dy/dx = y   [IN CLASS]

C&J defines the exponential function as the solution of the differential equation

dy/dx = y,  y(0) = 1

PREDICT (on paper, before running anything):
- If y grows faster whenever y is bigger, what shape do you expect the graph to have?
- Can y ever be zero or negative, starting from y(0) = 1? Why not?
- Is y increasing everywhere? How do you know just from the equation, without solving it?


In [ ]:
# Euler method for dy/dx = y, y(0) = 1
def euler_exp(h, x_max=3.0):
    xs = np.arange(0.0, x_max + h, h)
    ys = np.empty_like(xs)
    ys[0] = 1.0
    for i in range(1, len(xs)):
        ys[i] = ys[i - 1] + h * ys[i - 1]
    return xs, ys

h = 0.1
xs, ys_euler = euler_exp(h)
ys_exact = np.exp(xs)

plt.plot(xs, ys_euler, 'o-', label=f'Euler, h={h}')
plt.plot(xs, ys_exact, '-', label='np.exp(x)')
plt.xlabel('x'); plt.ylabel('y')
plt.title('Solving dy/dx = y, y(0) = 1')
plt.legend()
plt.show()


VERIFY: The two curves above should be nearly on top of each other. Zoom your eyes on the gap near x = 3. Which curve is above the other, and does that match how the Euler method systematically rounds off a convex function?


In [ ]:
plt.plot(xs, ys_euler - ys_exact, 'o-', color='crimson')
plt.axhline(0, color='black', linewidth=0.8)
plt.xlabel('x'); plt.ylabel('Euler error, y_euler minus exp(x)')
plt.title('Euler approximation error grows with x')
plt.show()


PREDICT: exp was defined only by a differential equation and an initial condition, nothing else. From that definition alone, C&J shows exp(x+y) = exp(x) times exp(y). Before running the next cell, write down why that is a nontrivial thing to prove (it does not follow just from looking at a graph).


In [ ]:
# Check the functional equation exp(x + y) = exp(x) * exp(y) numerically
x_vals = rng.uniform(-2, 2, size=5)
y_vals = rng.uniform(-2, 2, size=5)

for x, y in zip(x_vals, y_vals):
    lhs = np.exp(x + y)
    rhs = np.exp(x) * np.exp(y)
    print(f'x={x:+.3f}, y={y:+.3f}:  exp(x+y)={lhs:.6f}   exp(x)*exp(y)={rhs:.6f}   difference={lhs - rhs:.2e}')


REFLECT: In lecture we proved exp(x+y) = exp(x) exp(y) from uniqueness of solutions to dy/dx = y, not by manipulating a series or an integral. Where else in this course have we relied on an existence-and-uniqueness argument rather than an explicit formula?


### Extension: how good is the Euler approximation?   [HOMEWORK]

The plot above shows the Euler method drifting away from exp(x) as x grows. Let us measure exactly how fast that error shrinks as we shrink the step size h. This connects directly to the order of magnitude ideas from Section 3.7.


In [ ]:
hs = np.array([0.5, 0.2, 0.1, 0.05, 0.02, 0.01])
max_err = []
for h in hs:
    xs_h, ys_h = euler_exp(h)
    max_err.append(np.max(np.abs(ys_h - np.exp(xs_h))))
max_err = np.array(max_err)

plt.loglog(hs, max_err, 'o-', label='Euler max error on [0,3]')
plt.loglog(hs, hs, '--', label='reference line, slope 1')
plt.xlabel('step size h'); plt.ylabel('max error')
plt.title('Euler method error versus step size')
plt.legend()
plt.show()

# TODO: fit a line to the log-log data with:
#   slope, intercept = np.polyfit(np.log(hs), np.log(max_err), 1)
# What power of h does the slope tell you the error is proportional to?
# In the language of Section 3.7, write the error as O(h^?) and justify the exponent.


## Part 2: ln as the inverse of exp   [IN CLASS]

We define ln as the inverse function of exp: ln(x) is the unique number y such that exp(y) = x.

PREDICT: exp maps all real numbers onto the positive reals, strictly increasing. Sketch, on paper, what its inverse function must look like, including its domain and range, before running the cell below.


In [ ]:
# We already built a bisection root-finder back in Week 2 for the Intermediate Value Theorem.
# Inverting exp is the same idea: solve exp(x) = target for x.
def my_ln(target, lo=-50.0, hi=50.0):
    return brentq(lambda x: np.exp(x) - target, lo, hi)

test_vals = [0.5, 1.0, 2.0, 7.389, 100.0]
for v in test_vals:
    print(f'target={v:8.3f}   my_ln={my_ln(v):.6f}   np.log={np.log(v):.6f}')


In [ ]:
xs_all = np.linspace(-2, 3, 400)
xs_pos = np.linspace(0.05, 20, 400)

plt.plot(xs_all, np.exp(xs_all), label='exp(x)')
plt.plot(xs_pos, np.log(xs_pos), label='ln(x)')
plt.plot(xs_all, xs_all, 'k--', linewidth=0.8, label='y = x')
plt.xlim(-2, 8); plt.ylim(-2, 8)
plt.gca().set_aspect('equal')
plt.legend()
plt.title('exp and ln are reflections of each other across y = x')
plt.show()


VERIFY: In lecture we derived the log laws (ln(ab) = ln a + ln b, and so on) from the exponential laws, not independently. The next cell checks this numerically for random a, b, r.


In [ ]:
a_vals = rng.uniform(0.1, 10, size=5)
b_vals = rng.uniform(0.1, 10, size=5)
r_vals = rng.uniform(-3, 3, size=5)

for a, b, r in zip(a_vals, b_vals, r_vals):
    print(f'a={a:.3f} b={b:.3f} r={r:+.3f}')
    print(f'  ln(ab)    = {np.log(a * b):.6f}   ln a + ln b = {np.log(a) + np.log(b):.6f}')
    print(f'  ln(a/b)   = {np.log(a / b):.6f}   ln a - ln b = {np.log(a) - np.log(b):.6f}')
    print(f'  ln(a**r)  = {np.log(a ** r):.6f}   r * ln a    = {r * np.log(a):.6f}')


### Extension: derivative of ln and general powers   [HOMEWORK]

In lecture we derived d/dx ln(x) = 1/x from implicit differentiation applied to exp(ln x) = x, using the chain rule from Week 8. We also defined general powers by x^r = exp(r ln x). Verify both numerically.


In [ ]:
def central_diff(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

print('Derivative of ln:')
for x in [0.5, 1.0, 2.0, 5.0]:
    d_numeric = central_diff(np.log, x)
    print(f'  x={x:.2f}   numeric derivative = {d_numeric:.6f}   1/x = {1 / x:.6f}')

print()
print('General powers, x**r versus exp(r * ln x):')
r = np.pi
for x in [1.5, 2.0, 4.0]:
    lhs = x ** r
    rhs = np.exp(r * np.log(x))
    print(f'  x={x:.2f}   x**pi = {lhs:.6f}   exp(pi * ln x) = {rhs:.6f}')

# TODO: verify d/dx (x**r) = r * x**(r - 1) using central_diff on the
# function lambda x: x ** r, for r = pi and a couple of x values of your choice.


## Part 3: Applications of the exponential function   [IN CLASS]

Sections 3.4b through 3.4f describe six physical systems that are all governed by the exact same differential equation as Part 1, just with different signs and constants: dy/dx = k y. We will build one reusable model function and use it twice in class; four more applications of the same idea are the homework extension below.


In [ ]:
# One model for every first-order decay or growth process: y(t) = A * exp(-k * t)
def exp_model(t, A, k):
    return A * np.exp(-k * t)

# Application 1: radioactive decay (synthetic noisy measurements)
rng2 = np.random.default_rng(7)
t_data = np.linspace(0, 20, 15)
A_true, k_true = 100.0, 0.15
y_clean = exp_model(t_data, A_true, k_true)
y_data = y_clean + rng2.normal(scale=3.0, size=t_data.shape)

(A_fit, k_fit), _ = curve_fit(exp_model, t_data, y_data, p0=[80, 0.1])
half_life = np.log(2) / k_fit

t_fine = np.linspace(0, 20, 200)
plt.scatter(t_data, y_data, label='measured activity')
plt.plot(t_fine, exp_model(t_fine, A_fit, k_fit), 'r-', label=f'fit: A={A_fit:.1f}, k={k_fit:.3f}')
plt.xlabel('time'); plt.ylabel('activity')
plt.legend()
plt.title(f'Radioactive decay fit, half-life about {half_life:.2f}')
plt.show()

print(f'Fitted k = {k_fit:.4f}, half-life = ln(2)/k = {half_life:.3f}')


Application 2 uses the same equation again, this time from an RC circuit. Instead of fitting data, we solve the differential equation numerically with solve_ivp and compare to the closed-form solution derived in lecture.


In [ ]:
R, C = 1000.0, 1e-3  # ohms, farads; RC = 1 second here
RC = R * C
Vs = 5.0  # source voltage

def rc_charging(t, V):
    return [(Vs - V[0]) / RC]

sol = solve_ivp(rc_charging, [0, 5 * RC], [0.0], dense_output=True, max_step=RC / 50)
t_plot = np.linspace(0, 5 * RC, 200)
V_numeric = sol.sol(t_plot)[0]
V_exact = Vs * (1 - np.exp(-t_plot / RC))

plt.plot(t_plot, V_numeric, 'o', markersize=3, label='solve_ivp')
plt.plot(t_plot, V_exact, '-', label='closed form: Vs (1 - exp(-t/RC))')
plt.xlabel('time (s)'); plt.ylabel('capacitor voltage (V)')
plt.legend()
plt.title('RC circuit charging')
plt.show()

print('max numeric minus exact difference =', np.max(np.abs(V_numeric - V_exact)))


In [ ]:
def rc_discharging(t, V):
    return [-V[0] / RC]

sol2 = solve_ivp(rc_discharging, [0, 5 * RC], [Vs], dense_output=True, max_step=RC / 50)
V_discharge_numeric = sol2.sol(t_plot)[0]

plt.plot(t_plot, V_numeric, label='charging')
plt.plot(t_plot, V_discharge_numeric, label='discharging')
plt.xlabel('time (s)'); plt.ylabel('capacitor voltage (V)')
plt.legend()
plt.title('RC circuit: charging versus discharging')
plt.show()


REFLECT: The decay fit and the RC circuit used the exact same mathematics, dy/dt proportional to y, in two completely different units and contexts. Name one more system from your own major that you would expect to follow this same equation.


### Extension: the rest of Section 3.4   [HOMEWORK, due before the Week 10 quiz]

Lecture on Day 2 covers all six applications in Section 3.4; class time only had room to build two of them by hand. Complete the remaining four below, reusing exp_model and curve_fit wherever it fits.


In [ ]:
# Continuously compounded interest: compare discrete compounding to the continuous limit
P, r_rate, t_years = 1000.0, 0.06, 10
ns = [1, 4, 12, 365, 10000]
for n in ns:
    A_n = P * (1 + r_rate / n) ** (n * t_years)
    print(f'n={n:6d} compounds per year -> A = {A_n:.4f}')
print(f'continuous compounding      -> A = {P * np.exp(r_rate * t_years):.4f}')

# TODO: make a plot of A(n) for n = 1 up to 10000, with a horizontal reference
# line at the continuous-compounding value, showing A(n) approaching it as n grows.


In [ ]:
# Newton cooling law: dT/dt = -k (T - T_env)
# Closed form: T(t) = T_env + (T0 - T_env) * exp(-k t)
#
# TODO:
# 1. A cup of coffee starts at T0 = 90 degrees C in a room at T_env = 22 degrees C.
#    Solve dT/dt = -k (T - T_env) with solve_ivp for k = 0.05 (1/min) over 60 minutes,
#    the same way Application 2 solved the RC circuit.
# 2. Overlay the closed-form solution and confirm they agree.
# 3. Generate noisy measured temperatures every 5 minutes from the closed-form solution,
#    then use curve_fit (adapt exp_model, remembering the T_env shift) to recover k
#    from the noisy data, the same way Application 1 recovered k for radioactive decay.


In [ ]:
# Barometric formula: P(h) = P0 * exp(-h / H)
# P0 = 101.3 kPa at sea level, scale height H is about 8500 meters.
#
# TODO:
# 1. Plot P(h) for h from 0 to 20000 meters.
# 2. Find the altitude at which pressure drops to half its sea-level value.
#    (For comparison, the summit of Mount Everest is about 8850 meters.)


In [ ]:
# A first-order chemical reaction has concentration C(t) = C0 * exp(-k t).
# TODO: reuse exp_model and curve_fit exactly as in Application 1 to fit a rate
# constant k to the concentration-versus-time data below, then plot the data
# together with your fitted curve.
t_rxn = np.array([0, 10, 20, 30, 40, 50])
C_rxn = np.array([1.00, 0.74, 0.55, 0.41, 0.30, 0.22])  # mol/L


## Final reflection

- The differential-equation definition of exp gave us the derivative rule, the functional equation, and six physical applications, all from one starting point: dy/dx = y. Compare that to just being told the derivative of e^x is e^x. What did the construction buy you that memorization did not?
- Every application in Part 3 and its homework extension is the same equation wearing a different costume: money, radioactivity, temperature, pressure, voltage, concentration. Where do you expect to meet dy/dx = k y again later in your engineering coursework?
